In [2]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

/home/ethanlmines/blue_dir/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "openai/gpt-oss-20b"

model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 69917.08it/s]


In [1]:
system_prompt = r"""
Answer with one topic and quotation from the transcript per line.
The topic should be 1 to 30 characters.
The substring should be 1 to 256 characters.
Separate the two with a colon.
Do not add quote marks to your substring
"""
# Here's the format:
# topic a : transcript substring a
# topic b : transcript substring b
# topic c : transcript substring c

orig_topics = ["politics", "business", "sports", "entertainment"]
standard_messages = [
    # {"role": "system", "content": "Only answer in comma-delimited lists with no quotations:\na,b,c."},
    {"role": "system", "content": system_prompt},
]

In [ ]:
from tqdm import tqdm
import csv
import glob
import os

data_dir = "out/resegmented/"

file_names = []
texts = []

# Sample one episode from each show
show_dirs = [d_path for d_path in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d_path)]
for show_dir in tqdm(show_dirs, desc="Processing shows"):
    for csv_path in glob.glob(os.path.join(show_dir, "*.csv")):
        with open(csv_path, 'r') as r:
            texts.append("".join(row['text'] for row in csv.DictReader(r)))
        file_names.append(csv_path)

        # Debug statement
        break

Processing shows:   0%|          | 0/100 [00:00<?, ?it/s]

Processing shows: 100%|██████████| 100/100 [00:00<00:00, 579.10it/s]


In [8]:
import re
output_pattern = re.compile('([a-z ]+) : (.+)')


In [9]:
import xgrammar as xgr
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer)

# grammar = xgr.GrammarCompiler(tokenizer_info).compile_grammar(r"""
# root ::=  () | topic ("," topic){0,2}
# topic ::= [a-z][a-z ]{0,29}
# """)

grammar = xgr.GrammarCompiler(tokenizer_info).compile_grammar(r"""
root ::=  () | entry ("\n" entry){0,2}
entry ::= [a-z][a-z ]{0,29} " : " [^\r\n]{1,512}
topic ::= [a-z][a-z ]{0,29}
""")


In [12]:

out_rows = [

]

cur_topics = sorted(orig_topics)
set_topics = set(cur_topics)
i = 0
for file_path, text in zip(file_names, texts):
    user_prefix = f"""
You are tasked with performing topic modeling over podcast transcripts.
Here are the topics you've discovered so far:
{','.join(cur_topics)}

Given a podcast transcript, generate a list of no more than 3 topics.
Each topic should be on a separate line and be followed by a colon
and a substring of the transcript as your justification for the topic.

Consider the sample podcast transcript:
\"I really want to take a Mediterranean cruise. There's so much history on the coasts of Greece and Turkey. And we can enjoy some good ouzo.\"

Your output could be as follows:
travel : want to take a Mediterranean cruise
history : There's so much history
food and drink : enjoy some good ouzo

Now here's the actual transcript:
"""


    tokenized_prompt = tokenizer.apply_chat_template(standard_messages + [
        {"role": "user", "content": user_prefix + "\n" + text}
    ],
    tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
    output = model.generate(**tokenized_prompt,
                            logits_processor=[xgr.contrib.hf.LogitsProcessor(grammar)],
                            max_new_tokens=2048
    )
    input_length = tokenized_prompt['input_ids'].shape[-1]
    decoded = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)

    # The quote they give must be from the text itself--avoid hallucinations
    matches = output_pattern.findall(decoded)
    print(matches)
    new_topics = [ (topic, quote) for (topic, quote) in matches if quote in text]
    set_topics = set_topics | {t for t, _ in new_topics}
    cur_topics = sorted(set_topics)

    for topic, quote in new_topics:
        out_rows.append(
            (file_path, topic, quote)
        )
print(out_rows)


[('travel', 'started at a temple and a castle called Gnosis'), ('politics', '3% of government positions are held by women'), ('religion', 'Kundalini is a Sanskrit word and it is a word for goddess consciousness')]
[('politics', "I was aware of my opponent's strategy. And I wasn't about to fall prey or fall into those traps."), ('travel', "I really want to take a Mediterranean cruise. There's so much history on the coasts of Greece and Turkey."), ('food and drink', 'enjoy some good ouzo, supportafterabortion.com/iheart-compassion, supportafterabortion.com/iheart-hope, supportafterabortion.com/ihart-hope, emeraldmedcbd.com, promo code NUN, N-U-N-N, emeraldmedcbd.com, promo code NUN, N-U-N-N, emeraldmedcbd.com, promo code NUN, N-U-N-N, emeraldmedcbd.com, promo code NUN, N-U-N-N, emeraldmedcbd.com, promo code NUN, N-U-N-N, emeraldmedcbd.com, promo code NUN, N-U-N-N, emeraldmedcbd.com, promo code NUN, N-U-N-N, emeraldmedcbd.com, promo code NUN, N-U-N-N, emeraldmedcbd.com, ')]
[('business', 

In [13]:
for row in out_rows:
    print(row)

('resegment_out/6365708/6365708_00007.mp3.csv', 'travel', 'started at a temple and a castle called Gnosis')
('resegment_out/6365708/6365708_00007.mp3.csv', 'politics', '3% of government positions are held by women')
('resegment_out/6365708/6365708_00007.mp3.csv', 'religion', 'Kundalini is a Sanskrit word and it is a word for goddess consciousness')
('resegment_out/5929292/5929292_00022.mp3.csv', 'politics', "I was aware of my opponent's strategy. And I wasn't about to fall prey or fall into those traps.")
('resegment_out/729674/729674_00032.mp3.csv', 'business', 'making work to be a space for the space')
('resegment_out/729674/729674_00032.mp3.csv', 'entertainment', 'had a couple of shows in some regional galleries')
('resegment_out/1385199/1385199_00004.mp3.csv', 'politics', 'I was told that average rent is $900')
('resegment_out/1385199/1385199_00004.mp3.csv', 'real estate', "You can trade it, instead of selling it, you're trading it into another building")
('resegment_out/1317513/13

In [17]:
import csv
fields = ["episode_file", "topic", "episode_quote"]
out_path = "llama_generated_topics.csv"
with open(out_path, 'w') as w:
    writer = csv.writer(w)
    writer.writerow(fields)
    writer.writerows(out_rows)